In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash  # Import JupyterDash for running Dash in Jupyter Notebook

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl  # Import for geolocation mapping
from dash import dcc  # Import Dash core components for interactivity
from dash import html  # Import Dash HTML components for layout
import plotly.express as px  # Import for creating interactive plots
from dash import dash_table  # Import for creating data tables
from dash.dependencies import Input, Output, State  # Import for callback dependencies
import base64  # Import to encode the logo image

# Configure OS routines
import os  # Import for operating system-related functions (e.g., file path handling)

# Configure the plotting routines
import numpy as np  # Import for numerical operations
import pandas as pd  # Import for data manipulation and analysis
import matplotlib.pyplot as plt  # Import for basic plotting (though not used here directly)


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter  # Import the AnimalShelter class from the CRUD module

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"  # Define username for MongoDB authentication
password = "SNHU1234"  # Define password for MongoDB authentication

# Connect to database via CRUD Module
db = AnimalShelter(username, password)  # Instantiate AnimalShelter with credentials to connect to MongoDB

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))  # Convert MongoDB data to a Pandas DataFrame using all records

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True)  # Remove '_id' column to avoid data table crash, modify dataframe in place

## Debug
# print(len(df.to_dict(orient='records')))  # Uncomment to debug: print number of records
# print(df.columns)  # Uncomment to debug: print column names


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)  # Initialize the JupyterDash application

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'  # Define the correct logo file name with space
if not os.path.exists(image_filename):  # Check if file exists in current directory
    image_filename = os.path.join(os.path.expanduser('~'), 'Desktop', 'Grazioso Salvare Logo.png')  # Fallback to desktop path
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())  # Encode the logo image in base64
except FileNotFoundError:
    print(f"Error: {image_filename} not found. Please ensure 'Grazioso Salvare Logo.png' is on the desktop or in the current directory.")  # Error message if file not found
    encoded_image = None  # Set to None to avoid crash if file is missing

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([  # Define the main dashboard layout
    html.Center(html.B(html.H1('Grazioso Salvare Rescue Dashboard - Developed by Karen Connors'))),  # Center the title with developer name
    html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()) if encoded_image else '', style={'height': '64px'}), href="http://www.snhu.edu") if encoded_image else html.P("Logo not available"),  # Add logo with link, fallback if missing
    html.Hr(),  # Horizontal rule for separation
    html.Div(  # Container for the dropdown filter
        dcc.Dropdown(
            id='filter-type',  # ID for callback reference
            options=[  # Define filter options
                {'label': 'Reset', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'}
            ],
            value='all',  # Default selection
            style={'width': '50%', 'marginBottom': '20px'}  # Styling for dropdown
        )
    ),
    html.Hr(),  # Horizontal rule for separation
    dash_table.DataTable(id='datatable-id',  # Interactive data table
                        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],  # Define columns
                        data=df.to_dict('records'),  # Initial data
                        page_size=10,  # Pagination limit
                        sort_action='native',  # Enable native sorting
                        style_table={'overflowX': 'auto', 'maxHeight': '300px'}  # Styling for table
                       ),
    html.Br(),  # Line break
    html.Hr(),  # Horizontal rule for separation
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',  # Row container for side-by-side layout
             style={'display': 'flex'},  # Use flexbox for layout
             children=[  # Children divs for chart and map
        html.Div(
            id='graph-id',  # ID for pie chart
            className='col s12 m6',  # Bootstrap column class
        ),
        html.Div(
            id='map-id',  # ID for geolocation map
            className='col s12 m6',  # Bootstrap column class
        )
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id', 'data'),  # Output to update table data
              [Input('filter-type', 'value')])  # Input from dropdown
def update_dashboard(filter_type):  # Function to update dashboard based on filter
    # FIX ME Add code to filter interactive data table with MongoDB queries
    if filter_type == 'all':  # Reset to all data
        df = pd.DataFrame.from_records(db.read({}))
    else:  # Apply filter based on selected rescue type
        df = pd.DataFrame.from_records(db.filterData(filter_type))
    df.drop(columns=['_id'], inplace=True)  # Remove _id to avoid crash
    return df.to_dict('records')  # Return updated data

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(  # Callback to update pie chart
    Output('graph-id', "children"),  # Output for chart children
    [Input('datatable-id', "derived_virtual_data")])  # Input from table data
def update_graphs(viewData):  # Function to generate pie chart
    # FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if viewData is None:  # Handle case where no data is selected
        dff = df
    else:  # Use derived virtual data
        dff = pd.DataFrame.from_dict(viewData)
    pie_fig = px.pie(dff, names='breed', title='Preferred Animals')  # Create pie chart of breeds
    return [dcc.Graph(figure=pie_fig)]  # Return graph component

#This callback will highlight a cell on the data table when the user selects it
@app.callback(  # Callback to style selected columns
    Output('datatable-id', 'style_data_conditional'),  # Output for table styling
    [Input('datatable-id', 'selected_columns')])  # Input from selected columns
def update_styles(selected_columns):  # Function to apply styling
    # Fix: Handle None case for selected_columns
    if selected_columns is None:  # Check if no columns are selected
        return []  # Return empty list to avoid error
    return [{  # Return list of style conditions
        'if': {'column_id': i},  # Condition for specific column
        'background_color': '#D2F3FF'  # Highlight color
    } for i in selected_columns]  # Apply to each selected column

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(  # Callback to update geolocation map
    Output('map-id', "children"),  # Output for map children
    [Input('datatable-id', "derived_virtual_data"),  # Input from table data
     Input('datatable-id', "derived_virtual_selected_rows")])  # Input from selected rows
def update_map(viewData, index):  # Function to update map
    if viewData is None or index is None:  # Check for missing data
        return [dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[dl.TileLayer(id="base-layer-id")])]

    dff = pd.DataFrame.from_dict(viewData)  # Convert data to DataFrame
    row = index[0] if index else 0  # Get row index from selection, default to 0 if none
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),  # Base map layer
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]], children=[  # Marker at selected location
                dl.Tooltip(dff.iloc[row, 4]),  # Tooltip with breed
                dl.Popup([  # Popup with animal name
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row, 9])
                ])
            ])
        ])
    ]

app.run_server(debug=True, port=8050)  # Start the Dash server in debug mode with a specific port to avoid conflicts

Dash app running on http://127.0.0.1:8050/
